# MultiMed — Test evaluation

Notebook chạy pipeline trên 500 mẫu **test**, đánh giá NER gold và phân tích lỗi.

Kết quả được lưu riêng trong `/content/multimed_results_500_test_rerun/`.
Chọn **Runtime → Change runtime type → T4 GPU** rồi chạy các cell theo thứ tự.


In [ ]:
!pip -q install -U accelerate safetensors librosa soundfile jiwer seqeval

In [ ]:
import csv
import json
import time
from collections import Counter
from pathlib import Path

import torch
from datasets import Audio, load_dataset
from jiwer import wer
from transformers import (
    AutoModelForSpeechSeq2Seq,
    AutoModelForTokenClassification,
    AutoProcessor,
    AutoTokenizer,
    pipeline,
)

assert torch.cuda.is_available(), (
    'Chưa thấy GPU. Chọn Runtime → Change runtime type → T4 GPU.'
)
DEVICE = 'cuda'
DEVICE_INDEX = 0
TORCH_DTYPE = torch.float16
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Cấu hình test batch
ASR_REPO = 'leduckhai/MultiMed-ST'
ASR_SUBFOLDER = 'asr/whisper-small-vietnamese/checkpoint-5000'
ASR_PROCESSOR_SUBFOLDER = 'asr/whisper-small-vietnamese'
NER_REPO = 'leduckhai/VietMed-NER'
NER_SUBFOLDER = 'xlm-roberta-base-VietMed-NER'
DATASET_ID = 'leduckhai/VietMed'
SPLIT = 'test'
START_INDEX = 0
NUM_SAMPLES = 500
MAX_DURATION_SECONDS = 30
OUTPUT_DIR = Path('/content/multimed_results_500_test_rerun')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = OUTPUT_DIR / 'results.jsonl'
ERROR_PATH = OUTPUT_DIR / 'errors.jsonl'
print(f'Chạy {NUM_SAMPLES} mẫu từ {DATASET_ID}/{SPLIT}, bắt đầu tại index {START_INDEX}')
print(f'Kết quả tạm: {CHECKPOINT_PATH}')

In [ ]:
# Load model một lần cho toàn batch
asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_REPO, subfolder=ASR_SUBFOLDER, dtype=TORCH_DTYPE,
    low_cpu_mem_usage=True, use_safetensors=True,
).to(DEVICE)
asr_processor = AutoProcessor.from_pretrained(ASR_REPO, subfolder=ASR_PROCESSOR_SUBFOLDER)
asr_pipe = pipeline(
    'automatic-speech-recognition', model=asr_model,
    tokenizer=asr_processor.tokenizer, feature_extractor=asr_processor.feature_extractor,
    torch_dtype=TORCH_DTYPE, device=DEVICE_INDEX,
)

ner_model = AutoModelForTokenClassification.from_pretrained(
    NER_REPO, subfolder=NER_SUBFOLDER,
).to(DEVICE)
ner_tokenizer = AutoTokenizer.from_pretrained(NER_REPO, subfolder=NER_SUBFOLDER)
ner_pipe = pipeline(
    'ner', model=ner_model, tokenizer=ner_tokenizer,
    aggregation_strategy='simple', device=DEVICE_INDEX,
)
print('ASR và NER models loaded')

In [ ]:
# Lấy các mẫu audio cần chạy; streaming tránh tải toàn bộ dataset
# Vòng lặp lấy đủ NUM_SAMPLES mẫu hợp lệ, không chỉ giới hạn trong START_INDEX + NUM_SAMPLES.
dataset = load_dataset(DATASET_ID, split=SPLIT, streaming=True)
dataset = dataset.cast_column('audio', Audio(sampling_rate=16000, decode=True))
rows = []
target_start = START_INDEX
index = -1
for index, row in enumerate(dataset):
    if index < target_start:
        continue
    duration = row.get('duration')
    if duration is not None and MAX_DURATION_SECONDS and duration > MAX_DURATION_SECONDS:
        continue
    rows.append((index, row))
    if len(rows) >= NUM_SAMPLES:
        break
print('Số mẫu thực tế:', len(rows))
print('Index cuối đã nạp:', index)

In [ ]:
def extract_audio(row):
    import io
    import soundfile as sf

    audio = row['audio']
    if hasattr(audio, 'get_all_samples'):
        decoded = audio.get_all_samples()
        return decoded.data.squeeze().cpu().numpy(), int(decoded.sample_rate)
    if isinstance(audio, dict) and 'array' in audio and 'sampling_rate' in audio:
        return audio['array'], int(audio['sampling_rate'])
    if isinstance(audio, dict) and audio.get('bytes'):
        audio_array, audio_rate = sf.read(io.BytesIO(audio['bytes']), dtype='float32')
        if getattr(audio_array, 'ndim', 1) > 1:
            audio_array = audio_array.mean(axis=1)
        return audio_array, int(audio_rate)
    raise ValueError('Không giải mã được audio của mẫu dataset')

def extract_entities(text):
    entities = []
    for item in ner_pipe(text):
        label = item.get('entity_group') or item.get('entity')
        if label in {'0', 'O', 'dum'}:
            continue
        entities.append({
            'text': item['word'], 'label': label,
            'score': round(float(item['score']), 6),
            'start': int(item['start']), 'end': int(item['end']),
        })
    return entities

results_by_index = {}
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                results_by_index[item['index']] = item
print('Đã có kết quả checkpoint:', len(results_by_index))

pending_rows = [(index, row) for index, row in rows if index not in results_by_index]
started = time.time()
with CHECKPOINT_PATH.open('a', encoding='utf-8') as result_file, ERROR_PATH.open('a', encoding='utf-8') as error_file:
    for completed, (index, row) in enumerate(pending_rows, start=1):
        try:
            audio_array, audio_rate = extract_audio(row)
            asr_result = asr_pipe(
                {'raw': audio_array, 'sampling_rate': audio_rate},
                generate_kwargs={'language': 'Vietnamese', 'task': 'transcribe'},
            )
            transcript = asr_result['text'].strip()
            entities = extract_entities(transcript)
            reference = row.get('text', '') or ''
            sample_result = {
                'index': index,
                'utterance_id': row.get('utterance_id', ''),
                'audio_name': row.get('audio_name', ''),
                'reference_transcript': reference,
                'transcript': transcript,
                'wer': round(float(wer(reference, transcript)), 6) if reference and transcript else None,
                'duration': row.get('duration'),
                'entities': entities,
            }
            result_file.write(json.dumps(sample_result, ensure_ascii=False) + '\n')
            result_file.flush()
            results_by_index[index] = sample_result
            print('[{}/{}] index={} entities={} WER={}'.format(
                completed, len(pending_rows), index, len(entities), sample_result['wer']))
        except Exception as exc:
            error = {'index': index, 'error': repr(exc)}
            error_file.write(json.dumps(error, ensure_ascii=False) + '\n')
            error_file.flush()
            print('Bỏ qua index={}: {}'.format(index, exc))

results = [results_by_index[index] for index, _ in rows if index in results_by_index]
print('Hoàn tất phiên này sau {:.1f} giây; tổng kết quả: {}'.format(
    time.time() - started, len(results)))

In [ ]:
# Tổng hợp toàn bộ checkpoint thành JSONL, CSV và summary
results_by_index = {}
if CHECKPOINT_PATH.exists():
    with CHECKPOINT_PATH.open(encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                results_by_index[item['index']] = item
results = [results_by_index[index] for index in sorted(results_by_index)]

jsonl_path = OUTPUT_DIR / 'results.jsonl'
jsonl_path.write_text(
    ''.join(json.dumps(result, ensure_ascii=False) + '\n' for result in results),
    encoding='utf-8',
)
csv_path = OUTPUT_DIR / 'summary.csv'
with csv_path.open('w', encoding='utf-8-sig', newline='') as f:
    fields = ['index', 'utterance_id', 'audio_name', 'duration', 'wer', 'transcript', 'entity_count', 'entities']
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for result in results:
        writer.writerow({
            'index': result['index'],
            'utterance_id': result['utterance_id'],
            'audio_name': result['audio_name'],
            'duration': result['duration'],
            'wer': result['wer'],
            'transcript': result['transcript'],
            'entity_count': len(result['entities']),
            'entities': json.dumps(result['entities'], ensure_ascii=False),
        })

all_labels = Counter(entity['label'] for result in results for entity in result['entities'])
valid_wers = [result['wer'] for result in results if result['wer'] is not None]
summary = {
    'num_results': len(results),
    'num_errors': sum(1 for line in ERROR_PATH.read_text(encoding='utf-8').splitlines() if line.strip()) if ERROR_PATH.exists() else 0,
    'mean_wer': round(sum(valid_wers) / len(valid_wers), 6) if valid_wers else None,
    'median_wer': round(sorted(valid_wers)[len(valid_wers) // 2], 6) if valid_wers else None,
    'zero_wer_count': sum(value == 0 for value in valid_wers),
    'total_entities': sum(len(result['entities']) for result in results),
    'entity_label_counts': dict(all_labels),
    'asr_model': f'{ASR_REPO}/{ASR_SUBFOLDER}',
    'ner_model': f'{NER_REPO}/{NER_SUBFOLDER}',
}
summary_path = OUTPUT_DIR / 'summary.json'
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(summary, ensure_ascii=False, indent=2))
print('Đã lưu:', jsonl_path, csv_path, summary_path)

In [ ]:
# Giai đoạn 3 — so sánh NER trên transcript chuẩn và transcript ASR
# Đây là phép đo nhất quán của cùng một model NER, không phải Precision/Recall/F1 gold.
from collections import Counter

comparison_results = []
for result in results:
    reference_entities = extract_entities(result['reference_transcript'])
    asr_entities = result['entities']
    reference_keys = Counter((item['text'].strip().lower(), item['label']) for item in reference_entities)
    asr_keys = Counter((item['text'].strip().lower(), item['label']) for item in asr_entities)
    matched = sum((reference_keys & asr_keys).values())
    reference_count = sum(reference_keys.values())
    asr_count = sum(asr_keys.values())
    comparison_results.append({
        'index': result['index'],
        'wer': result['wer'],
        'reference_entity_count': reference_count,
        'asr_entity_count': asr_count,
        'matched_entity_count': matched,
        'reference_entities': reference_entities,
        'asr_entities': asr_entities,
        'reference_only': [dict(text=text, label=label, count=count) for (text, label), count in (reference_keys - asr_keys).items()],
        'asr_only': [dict(text=text, label=label, count=count) for (text, label), count in (asr_keys - reference_keys).items()],
    })

comparison_path = OUTPUT_DIR / 'ner_comparison.jsonl'
comparison_path.write_text(
    ''.join(json.dumps(item, ensure_ascii=False) + '\n' for item in comparison_results),
    encoding='utf-8',
)
comparison_csv_path = OUTPUT_DIR / 'ner_comparison.csv'
with comparison_csv_path.open('w', encoding='utf-8-sig', newline='') as f:
    fields = ['index', 'wer', 'reference_entity_count', 'asr_entity_count', 'matched_entity_count', 'reference_only', 'asr_only']
    writer = csv.DictWriter(f, fieldnames=fields)
    writer.writeheader()
    for item in comparison_results:
        writer.writerow({
            'index': item['index'],
            'wer': item['wer'],
            'reference_entity_count': item['reference_entity_count'],
            'asr_entity_count': item['asr_entity_count'],
            'matched_entity_count': item['matched_entity_count'],
            'reference_only': json.dumps(item['reference_only'], ensure_ascii=False),
            'asr_only': json.dumps(item['asr_only'], ensure_ascii=False),
        })

reference_total = sum(item['reference_entity_count'] for item in comparison_results)
asr_total = sum(item['asr_entity_count'] for item in comparison_results)
matched_total = sum(item['matched_entity_count'] for item in comparison_results)
comparison_summary = {
    'num_results': len(comparison_results),
    'reference_entity_total': reference_total,
    'asr_entity_total': asr_total,
    'matched_entity_total': matched_total,
    'entity_retention_against_reference_predictions': round(matched_total / reference_total, 6) if reference_total else None,
    'asr_entity_precision_against_reference_predictions': round(matched_total / asr_total, 6) if asr_total else None,
    'asr_entity_recall_against_reference_predictions': round(matched_total / reference_total, 6) if reference_total else None,
    'asr_entity_f1_against_reference_predictions': round(2 * matched_total / (reference_total + asr_total), 6) if reference_total + asr_total else None,
    'warning': 'These are consistency metrics between model predictions on reference and ASR text, not gold NER metrics.',
}
comparison_summary_path = OUTPUT_DIR / 'ner_comparison_summary.json'
comparison_summary_path.write_text(json.dumps(comparison_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(comparison_summary, ensure_ascii=False, indent=2))
print('Đã lưu:', comparison_path, comparison_csv_path, comparison_summary_path)

## Giai đoạn 4 — đánh giá NER trên gold labels

Phần này dùng dataset chính thức `leduckhai/VietMed-NER`, có các trường `words`, `tags`, `labels` và `text`, để tính Precision, Recall và F1 chính thức cho model NER. Dataset này có các split `train`, `validation`, `test`; notebook dùng `test`.

In [ ]:
# Tải dataset NER chính thức có gold labels
NER_DATASET_ID = 'leduckhai/VietMed-NER'
NER_EVAL_SPLIT = 'test'
NER_OUTPUT_DIR = Path('/content/multimed_ner_gold')
NER_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

try:
    ner_dataset = load_dataset(NER_DATASET_ID, split=NER_EVAL_SPLIT)
except Exception as exc:
    raise RuntimeError(
        'Không tải được dataset chính thức leduckhai/VietMed-NER. '
        'Nếu Hugging Face yêu cầu quyền truy cập, chạy huggingface_hub.login() trước cell này.'
    ) from exc

print(ner_dataset)
print('Columns:', ner_dataset.column_names)
print('Features:', ner_dataset.features)

In [ ]:
# Chạy NER trên gold test set và tính entity-level metrics
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score, precision_score, recall_score

if 'words' not in ner_dataset.column_names or 'labels' not in ner_dataset.column_names:
    raise ValueError(f'Dataset phải có words/labels, nhận được: {ner_dataset.column_names}')

model_id_to_label = {int(k): v for k, v in ner_model.config.id2label.items()}
def normalize_tag(tag):
    # seqeval dùng O cho nhãn ngoài thực thể; dataset/model dùng 0.
    if tag in {'O', '0', 'dum'}:
        return 'O'
    return tag

gold_sequences = []
pred_sequences = []
for example in ner_dataset:
    words = example['words']
    gold_tags = [normalize_tag(tag) for tag in example['labels']]
    encoded = ner_tokenizer(
        words, is_split_into_words=True, truncation=True, return_tensors='pt'
    )
    word_ids = ner_tokenizer(
        words, is_split_into_words=True, truncation=True
    ).word_ids()
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    with torch.inference_mode():
        logits = ner_model(**encoded).logits[0]
    predicted_ids = logits.argmax(dim=-1).tolist()
    predicted_by_word = {}
    for token_id, word_id in zip(predicted_ids, word_ids):
        if word_id is not None and word_id not in predicted_by_word:
            predicted_by_word[word_id] = model_id_to_label[token_id]
    predicted_tags = [normalize_tag(predicted_by_word.get(i, 'O')) for i in range(len(words))]
    length = min(len(gold_tags), len(predicted_tags))
    gold_sequences.append(gold_tags[:length])
    pred_sequences.append(predicted_tags[:length])

gold_report = seqeval_report(gold_sequences, pred_sequences, digits=4, output_dict=True)
overall = {
    'precision': round(float(precision_score(gold_sequences, pred_sequences)) * 100, 4),
    'recall': round(float(recall_score(gold_sequences, pred_sequences)) * 100, 4),
    'f1': round(float(f1_score(gold_sequences, pred_sequences)) * 100, 4),
}
per_label = {
    label: {
        'precision': round(float(values['precision']) * 100, 4),
        'recall': round(float(values['recall']) * 100, 4),
        'f1': round(float(values['f1-score']) * 100, 4),
        'support': int(values['support']),
    }
    for label, values in gold_report.items()
    if isinstance(values, dict) and 'f1-score' in values
}
metrics_payload = {
    'dataset': NER_DATASET_ID,
    'split': NER_EVAL_SPLIT,
    'num_examples': len(gold_sequences),
    'model': f'{NER_REPO}/{NER_SUBFOLDER}',
    'overall': overall,
    'per_label': per_label,
    'note': 'Metrics compare predictions against official gold labels using the first-subword prediction for each word; O is the outside tag for seqeval.',
}
metrics_path = NER_OUTPUT_DIR / 'ner_gold_metrics.json'
metrics_path.write_text(
    json.dumps(metrics_payload, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(overall, ensure_ascii=False, indent=2))
print('Saved:', metrics_path)

In [ ]:
# Tải metrics gold về máy
from google.colab import files
files.download(str(metrics_path))

In [ ]:
# Tải các file kết quả về máy
from google.colab import files
files.download(str(jsonl_path))
files.download(str(csv_path))
files.download(str(OUTPUT_DIR / 'summary.json'))

## Giai đoạn 2 — cách đọc kết quả

- `WER` so sánh transcript ASR với trường `text` chuẩn của VietMed; WER càng thấp càng tốt.
- `results.jsonl` giữ toàn bộ output của từng audio test.
- `summary.csv` thuận tiện mở bằng Excel.
- `summary.json` chứa WER trung bình, trung vị, số mẫu WER bằng 0 và phân bố entity.
- Vì VietMed không chứa nhãn NER, số lượng entity chưa phải Precision/Recall/F1. Muốn tính F1 cần chạy trên dataset VietMed-NER có nhãn `words`/`tags`.
- Nếu gặp lỗi VRAM, giảm `NUM_SAMPLES` xuống 100 hoặc giảm `MAX_DURATION_SECONDS`.
- Kết quả của Giai đoạn 2 nằm trong `/content/multimed_results_500_test/` và không ghi đè kết quả train của Giai đoạn 1.

## Giai đoạn 5 — phân tích lỗi và tạo số liệu báo cáo

Phần này dùng lại kết quả Giai đoạn 2–3 để:

- Chia lỗi ASR theo nhóm WER.
- So sánh số entity giữa transcript chuẩn và transcript ASR.
- Tạo các case study có WER cao hoặc entity bị mất/phát sinh.
- Xuất CSV/JSON và biểu đồ để đưa vào báo cáo.

Các số liệu entity ở đây là dự đoán của model; chỉ số NER gold chính thức nằm ở Giai đoạn 4.

In [ ]:
# Tạo bảng phân tích theo nhóm WER
import pandas as pd

if 'comparison_results' not in globals():
    raise RuntimeError('Hãy chạy cell Giai đoạn 3 trước cell này.')

analysis_rows = []
for item in comparison_results:
    wer_value = item['wer']
    if wer_value == 0:
        wer_group = '0%'
    elif wer_value <= 0.10:
        wer_group = '0–10%'
    elif wer_value <= 0.30:
        wer_group = '10–30%'
    else:
        wer_group = '>30%'
    analysis_rows.append({
        'index': item['index'],
        'wer': wer_value,
        'wer_group': wer_group,
        'reference_entity_count': item['reference_entity_count'],
        'asr_entity_count': item['asr_entity_count'],
        'matched_entity_count': item['matched_entity_count'],
        'reference_only_count': sum(x['count'] for x in item['reference_only']),
        'asr_only_count': sum(x['count'] for x in item['asr_only']),
    })

analysis_df = pd.DataFrame(analysis_rows)
group_order = ['0%', '0–10%', '10–30%', '>30%']
analysis_df['wer_group'] = pd.Categorical(analysis_df['wer_group'], categories=group_order, ordered=True)
group_summary = analysis_df.groupby('wer_group', observed=False).agg(
    num_samples=('index', 'count'),
    mean_wer=('wer', 'mean'),
    reference_entities=('reference_entity_count', 'sum'),
    asr_entities=('asr_entity_count', 'sum'),
    matched_entities=('matched_entity_count', 'sum'),
    reference_only=('reference_only_count', 'sum'),
    asr_only=('asr_only_count', 'sum'),
).reset_index()
group_summary['retention'] = group_summary['matched_entities'] / group_summary['reference_entities'].replace(0, pd.NA)
for column in ['mean_wer', 'retention']:
    group_summary[column] = group_summary[column].astype(float).round(6)

analysis_df.to_csv(OUTPUT_DIR / 'error_analysis_by_sample.csv', index=False, encoding='utf-8-sig')
group_summary.to_csv(OUTPUT_DIR / 'error_analysis_by_wer_group.csv', index=False, encoding='utf-8-sig')
print(group_summary.to_string(index=False))
print('Đã lưu bảng phân tích lỗi.')

In [ ]:
# Chọn các case study để đưa vào báo cáo
case_studies = []
for item in sorted(comparison_results, key=lambda x: x['wer'], reverse=True)[:10]:
    case_studies.append({
        'index': item['index'],
        'wer': item['wer'],
        'reference_only': item['reference_only'],
        'asr_only': item['asr_only'],
        'reference_entities': item['reference_entities'],
        'asr_entities': item['asr_entities'],
    })
case_study_path = OUTPUT_DIR / 'case_studies.json'
case_study_path.write_text(json.dumps(case_studies, ensure_ascii=False, indent=2), encoding='utf-8')
print('Các case study WER cao nhất:')
for case in case_studies[:5]:
    print(f"index={case['index']}, WER={case['wer']}, mất={len(case['reference_only'])}, phát sinh={len(case['asr_only'])}")
print('Đã lưu:', case_study_path)

In [ ]:
# Tạo biểu đồ phục vụ báo cáo
import matplotlib.pyplot as plt

plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(group_summary['wer_group'].astype(str), group_summary['mean_wer'] * 100, color='#3B82F6')
axes[0].set_title('WER trung bình theo nhóm')
axes[0].set_xlabel('Nhóm WER')
axes[0].set_ylabel('WER (%)')

label_counts = Counter(entity['label'] for result in results for entity in result['entities'])
labels, counts = zip(*label_counts.most_common())
axes[1].barh(list(labels)[::-1], list(counts)[::-1], color='#14B8A6')
axes[1].set_title('Phân bố entity dự đoán trên ASR transcript')
axes[1].set_xlabel('Số entity')
axes[1].set_ylabel('Nhãn')

fig.tight_layout()
figure_path = OUTPUT_DIR / 'stage5_error_analysis.png'
fig.savefig(figure_path, dpi=180, bbox_inches='tight')
plt.show()
print('Đã lưu:', figure_path)

In [ ]:
# Lưu summary và tải các file Giai đoạn 5
stage5_summary = {
    'stage': 5,
    'num_samples': len(analysis_df),
    'wer_group_summary': group_summary.to_dict(orient='records'),
    'case_study_count': len(case_studies),
    'interpretation': {
        'reference_only': 'Entity model dự đoán trên transcript chuẩn nhưng không dự đoán trên transcript ASR.',
        'asr_only': 'Entity model dự đoán trên transcript ASR nhưng không dự đoán trên transcript chuẩn.',
        'warning': 'Đây là phân tích tác động ASR, không phải gold NER evaluation.'
    }
}
stage5_summary_path = OUTPUT_DIR / 'stage5_summary.json'
stage5_summary_path.write_text(json.dumps(stage5_summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('Đã lưu:', stage5_summary_path)

from google.colab import files
for path in [
    OUTPUT_DIR / 'error_analysis_by_sample.csv',
    OUTPUT_DIR / 'error_analysis_by_wer_group.csv',
    OUTPUT_DIR / 'case_studies.json',
    OUTPUT_DIR / 'stage5_summary.json',
    OUTPUT_DIR / 'stage5_error_analysis.png',
]:
    files.download(str(path))